In [0]:
bronze_path   = 'bikestore.bronze'
silver_path   = 'bikestore.silver'
gold_path     = 'bikestore.gold'
resource_path = "/Volumes/bikestore/resource/origem"
resource_path_volume = '/Volumes/bikestore/resource/origem'

In [0]:
# criar varias tabelas temporárias de forma prática

bronze_map = {
    #View_name --------------- path    
    "tmp_bronze_brands":      f"{bronze_path}.brands",
    "tmp_bronze_categories":  f"{bronze_path}.categories",
    # "tmp_bronze_customers":   f"{bronze_path}.customers",
    # "tmp_bronze_order_items": f"{bronze_path}.order_items",
    # "tmp_bronze_orders":      f"{bronze_path}.orders",
    "tmp_bronze_products":    f"{bronze_path}.products",
    # "tmp_bronze_staffs":      f"{bronze_path}.staffs",
    "tmp_bronze_stocks":      f"{bronze_path}.stocks",
    # "tmp_bronze_stores":      f"{bronze_path}.stores",
}
    
for view_name, path in bronze_map.items():
    (spark.table(path)
        .createOrReplaceTempView(view_name))

In [0]:
from pyspark.sql.functions import current_timestamp, lit

current_ts = current_timestamp()

df_product_silver = spark.sql("""
with stock as (
  SELECT
    product_id,
    sum(quantity) as total_stock,
    collect_set(bronze_source_file) as source_stocks_file
  from
    tmp_bronze_stocks
  GROUP BY
    product_id
)
select
  p.product_id,
  p.category_id,
  p.product_name,
  c.category_name,
  b.brand_name,
  s.total_stock,
  p.model_year,
  p.list_price,
  -- Metadata
  b.bronze_source_file as source_brands_file,
  p.bronze_source_file as source_products_file,
  s.source_stocks_file
-- p.brand_id,
-- b.brand_id as b_brand_id,
-- c.category_id as c_category_id,
from
  tmp_bronze_products as p
    LEFT JOIN tmp_bronze_categories as c
      ON p.category_id = c.category_id
    LEFT JOIN tmp_bronze_brands as b
      ON p.brand_id = b.brand_id
    LEFT JOIN stock as s
      ON p.product_id = s.product_id;
""")\
  .withColumn("ingestion_timestamp", current_ts) \
  .withColumn("created_at", current_ts) \
  .withColumn("updated_at", lit(None).cast("timestamp"))

In [0]:
#salvar em parquet como delta na camada silver
df_product_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .partitionBy("model_year")\
    .saveAsTable(f"{silver_path}.product")